In [13]:
import os
import faiss
import torch
import ollama
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer, util

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
ALPHA = 0.4 
OLLAMA_MODEL = "moondream" # Ensure you ran `ollama pull moondream`

MODELS = {}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def get_raw_tensor(output):
    """Accesses the tensor inside the HuggingFace wrapper safely."""
    if torch.is_tensor(output): return output
    for attr in ("image_embeds", "pooler_output", "last_hidden_state"):
        if hasattr(output, attr):
            return getattr(output, attr)
    try: return output[0]
    except: return output

# ---------------------------------------------------------------------------
# Stack Initialization
# ---------------------------------------------------------------------------

def initialize_stack():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"--- [INFO] Initializing Stack on: {device} ---")

    # 1. CLIP (Stage 1)
    clip_id = "openai/clip-vit-base-patch32"
    MODELS["clip_model"] = CLIPModel.from_pretrained(clip_id).to(device)
    MODELS["clip_processor"] = CLIPProcessor.from_pretrained(clip_id)

    # 2. SBERT (Stage 3)
    MODELS["text_model"] = SentenceTransformer("all-MiniLM-L6-v2")

    # 3. Detect CLIP dimension
    dummy = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        raw_out = MODELS["clip_model"].get_image_features(pixel_values=dummy)
        dummy_tensor = get_raw_tensor(raw_out)

    MODELS["dim"] = dummy_tensor.shape[-1]
    MODELS["device"] = device
    print(f"--- [SUCCESS] Models ready. CLIP dim: {MODELS['dim']} ---")

# ---------------------------------------------------------------------------
# Stage 2 – Identity Probing (Moondream Optimized)
# ---------------------------------------------------------------------------

def get_vlm_identity(path: str) -> str:
    """
    Uses Moondream to probe for 'Ontological Identity' vs 'Surface Traits'.
    """
    # Optimized prompt for Moondream to fight Neural Surface Bias
    prompt = (
        "Identify this object. Is it a real organic item or a manufactured prop/ornament? "
        "Look for artificial textures, seams, or unnatural gloss. Describe its material and nature."
    )

    try:
        # Use ollama.generate for single-shot VLM tasks (faster than .chat)
        response = ollama.generate(
            model=OLLAMA_MODEL,
            prompt=prompt,
            images=[path],
            options={"temperature": 0, "num_predict": 50} 
        )
        return response["response"].strip()

    except Exception as e:
        return f"[VLM Error] {e}"

initialize_stack()

--- [INFO] Initializing Stack on: cpu ---


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 29164.77it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6261.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


--- [SUCCESS] Models ready. CLIP dim: 512 ---


In [14]:
def run_hsvf_pipeline(query_path: str, gallery_dir: str) -> None:
    print(f"\n{'=' * 30} HSVF RETRIEVAL (TOP-{TOP_K} GATE) {'=' * 30}")
    
    # 1. Path Setup
    query_path = os.path.abspath(query_path)
    gallery_dir = os.path.abspath(gallery_dir)
    exts = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}
    gallery_paths = [os.path.join(gallery_dir, f) for f in os.listdir(gallery_dir) 
                     if os.path.isfile(os.path.join(gallery_dir, f)) and os.path.splitext(f.lower())[1] in exts]

    # ------------------------------------------------------------------
    # Stage 1: CLIP Coarse Filter
    # ------------------------------------------------------------------
    print(f"[1/3] Indexing {len(gallery_paths)} images with CLIP...")
    index = faiss.IndexFlatL2(MODELS["dim"])
    for p in gallery_paths:
        try: index.add(get_clip_features(p))
        except: continue

    q_emb = get_clip_features(query_path)
    search_limit = min(TOP_K, len(gallery_paths))
    distances, indices = index.search(q_emb, search_limit)

    # ------------------------------------------------------------------
    # Stage 2: Query Reasoning
    # ------------------------------------------------------------------
    print(f"\n[2/3] Probing Query Identity with {OLLAMA_MODEL}...")
    q_identity = get_vlm_identity(query_path)
    
    print("-" * 60)
    print(f"FULL QUERY REASONING:\n{q_identity}")
    print("-" * 60)
    
    q_text_emb = MODELS["text_model"].encode(q_identity, convert_to_tensor=True)

    # ------------------------------------------------------------------
    # Stage 3: Top-K Semantic Verification
    # ------------------------------------------------------------------
    results = []
    print(f"\n[3/3] Verifying Shortlist (Full Logic Check)...")
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        img_path = gallery_paths[idx]
        filename = os.path.basename(img_path)
        
        print(f"      [{rank+1}/{search_limit}] Analyzing: {filename}...", end="\r")

        # Get full identity reasoning
        img_identity = get_vlm_identity(img_path)
        
        # SBERT Alignment
        m_text_emb = MODELS["text_model"].encode(img_identity, convert_to_tensor=True)
        semantic_score = util.cos_sim(q_text_emb, m_text_emb).item()
        
        # Score Normalization
        visual_score = float(np.clip(1.0 - dist / 2.0, 0.0, 1.0))
        final_score = (ALPHA * visual_score) + ((1 - ALPHA) * semantic_score)

        results.append({
            "fn": filename, 
            "vr": rank + 1, 
            "sc": final_score, 
            "vs": visual_score, 
            "ss": semantic_score, 
            "full_desc": img_identity
        })

    # ------------------------------------------------------------------
    # Final Output: Full Reasoning Table
    # ------------------------------------------------------------------
    results.sort(key=lambda x: x["sc"], reverse=True)

    print(f"\n\n{'=' * 30} FINAL HSVF LEADERBOARD {'=' * 30}")
    
    for i, res in enumerate(results):
        # 1. Print Rank and Scores
        print(f"\n#{i+1} | {res['fn']}")
        print(f"   [FUSION SCORE: {res['sc']:.4f}]  (CLIP: {res['vs']:.4f} | SBERT: {res['ss']:.4f})")
        print(f"   Original Visual Rank: {res['vr']}")
        
        # 2. Print the FULL VLM Reasoning
        print(f"   VLM REASONING:")
        # Indent the reasoning so it's clearly separated
        indented_desc = "      " + res['full_desc'].replace("\n", "\n      ")
        print(indented_desc)
        print("-" * 80)

# ---------------------------------------------------------------------------
# Execution
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    if not MODELS:
        initialize_stack()
    run_hsvf_pipeline(query_path="./img/query_apple.jpg", gallery_dir="./img")


============================== HSVF RETRIEVAL (TOP-3 GATE) ==============================
[1/3] Indexing 9 images with CLIP...

[2/3] Probing Query Identity with moondream...
------------------------------------------------------------
FULL QUERY REASONING:

------------------------------------------------------------

[3/3] Verifying Shortlist (Full Logic Check)...
      [3/3] Analyzing: apple2.jpg...pg...

============================== FINAL HSVF LEADERBOARD ==============================

#1 | query_apple.jpg
   [FUSION SCORE: 1.0000]  (CLIP: 1.0000 | SBERT: 1.0000)
   Original Visual Rank: 1
   VLM REASONING:
      
--------------------------------------------------------------------------------

#2 | apple1.jpg
   [FUSION SCORE: 0.4171]  (CLIP: 0.9712 | SBERT: 0.0477)
   Original Visual Rank: 2
   VLM REASONING:
      The image features an apple with a single green leaf attached to the top of it. The apple is red in color and appears to be a real organic item rather than a manuf

In [16]:
if __name__ == "__main__":
    # Ensure stack is ready
    if not MODELS:
        initialize_stack()
    
    # Run the test
    # Ensure these paths are correct for your machine!
    run_hsvf_pipeline(query_path="./img/query_watch.jpg", gallery_dir="./img")


============================== HSVF RETRIEVAL (TOP-3 GATE) ==============================
[1/3] Indexing 9 images with CLIP...

[2/3] Probing Query Identity with moondream...
------------------------------------------------------------
FULL QUERY REASONING:
The image shows an elegant black wristwatch with a leather strap. The watch has a round face featuring white numbers and hands that are clearly visible against the dark background of the watch's design. The watch appears to be made from genuine leather, giving
------------------------------------------------------------

[3/3] Verifying Shortlist (Full Logic Check)...
      [3/3] Analyzing: compass_distractor.jpeg...

============================== FINAL HSVF LEADERBOARD ==============================

#1 | query_watch.jpg
   [FUSION SCORE: 1.0000]  (CLIP: 1.0000 | SBERT: 1.0000)
   Original Visual Rank: 1
   VLM REASONING:
      The image shows an elegant black wristwatch with a leather strap. The watch has a round face featuring 